# Red Neuronal Profunda

## 1. Mejoramiento del modelo para CIFAR10


### 1.1 Importa las librerías relevantes

In [ ]:
import tensorflow as tf
from tensorflow import keras
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os

### 1.2 Construye un DNN con 20 capas escondidas de 100 neuronas cada una

In [ ]:
# Cargamos CIFAR10
(x_full_train, y_full_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
# Semillas a 42
tf.random.set_seed(42)
np.random.seed(42)

# Limpia sesión por si colab tiene algo cargado
keras.backend.clear_session()

In [ ]:
# Modelo secuencial: las capas se apilan en orden,
# y la salida de una capa pasa directamente a la siguiente.
model = keras.models.Sequential()

# Flatten: Convierte cada imagen de entrada (32x32 píxeles, 3 canales RGB)
# en un vector unidimensional de 32*32*3 = 3072 elementos.
# Las capas densas solo aceptan vectores planos, no tensores 3D.
model.add(keras.layers.Flatten(input_shape = [32, 32, 3]))

# Este bucle agrega 20 capas idénticas. Cada capa es "Dense",
# es decir, totalmente conectada: cada neurona se conecta con
# todas las salidas de la capa anterior.
#
# La función de activación usada es ReLU (Rectified Linear Unit):
#   f(x) = max(0, x)
# Su función es introducir no linealidad para que el modelo
# aprenda relaciones complejas. Es rápida y evita que el gradiente
# desaparezca (problema común con sigmoid/tanh).
for _ in range(20):
    model.add(keras.layers.Dense(100, activation = "relu"))

# CAPA DE SALIDA: Dense(10, softmax)
# Esta capa produce la salida final del modelo. Tiene 10 neuronas,
# una por cada clase del conjunto CIFAR-10:
#   [avión, coche, pájaro, gato, ciervo, perro, rana, caballo, barco, camión]
#
# La activación "softmax" convierte los valores de salida (logits)
# en probabilidades que suman 1, con la fórmula:
#   softmax(z_i) = e^(z_i) / Σ_j e^(z_j)
model.add(keras.layers.Dense(10, activation = "softmax"))

# Red profunda con 20 capas ocultas y 10 salidas.
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
# Compilemos con parámetros básicos como sparse y SGD, con una métrica de exactitud simple
model.compile(loss = "sparse_categorical_crossentropy", optimizer = "sgd", metrics = ["accuracy"])

In [ ]:
# Vamos a utilizar early stopping asi que necesitamos dataset de validación
x_train = x_full_train[5000:]
y_train = y_full_train[5000:]
x_valid = x_full_train[:5000]
y_valid = y_full_train[:5000]

In [ ]:
# Entrenamiento del modelo
history = model.fit(x_train,y_train, epochs = 25, validation_data = (x_valid, y_valid))

Epoch 1/25
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.1600 - loss: 2.1831 - val_accuracy: 0.2618 - val_loss: 1.9270
Epoch 2/25
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.2735 - loss: 1.9256 - val_accuracy: 0.2822 - val_loss: 1.9005
Epoch 3/25
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.3182 - loss: 1.8458 - val_accuracy: 0.3274 - val_loss: 1.8230
Epoch 4/25
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.3428 - loss: 1.7928 - val_accuracy: 0.3262 - val_loss: 1.8306
Epoch 5/25
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.3614 - loss: 1.7496 - val_accuracy: 0.3756 - val_loss: 1.7125
Epoch 6/25
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.3817 - loss: 1.7051 - val_accuracy: 0.3540 - val_loss: 1.7556
Epoch 7/25
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.3975 - loss: 1.6720 - val_accuracy: 0.3234 - val_loss: 1.8882
Epoch 8/25
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.4084 - loss: 1.6436 

In [ ]:
# Evaluación del modelo
model.evaluate(x_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.4395 - loss: 1.6182


[1.6338255405426025, 0.4374000132083893]

### 1.3 Ahora prueba con inicializacion HE y la función de activación ELU

In [ ]:
# Limpieza de sesión
keras.backend.clear_session()

In [ ]:
# Semillas a 42
tf.random.set_seed(42)
np.random.seed(42)

In [ ]:
# Construcción del modelo secuencial
model = keras.models.Sequential()
model.add(keras.layers.Flatten(input_shape = [32,32,3]))
for _ in range(20):
    model.add(keras.layers.Dense(100, activation = "elu", kernel_initializer = "he_normal"))
model.add(keras.layers.Dense(10, activation = "softmax"))

#### Función de Activación ELU e Inicializador He_Normal en Redes Neuronales

---

## - Función de Activación: ELU (Exponential Linear Unit)

La función **ELU** (*Exponential Linear Unit*) es una mejora sobre la función **ReLU**, diseñada para evitar el problema de las **neuronas muertas** y mejorar la **estabilidad del entrenamiento** en redes profundas.

### Definición matemática

$$
f(x) =
\begin{cases}
x, & \text{si } x > 0 \\
\alpha(e^x - 1), & \text{si } x \le 0
\end{cases}
$$

donde $\alpha$ es un parámetro positivo (típicamente $\alpha = 1$).

---

### - Comportamiento comparado con ReLU

Región | ReLU | ELU
--------|:------|:------
$$x > 0$$ | $$f(x) = x$$ | $$f(x) = x$$
$$x \le 0$$ | $$f(x) = 0$$ | $$f(x) = \alpha(e^x - 1)$$

Para valores **positivos**, ambas son idénticas (lineales).  
Para valores **negativos**, **ELU** produce una salida **suavemente negativa** en lugar de cero.

---

### Ventajas de ELU

1. Evita neuronas muertas: incluso si $x < 0$, la derivada de ELU no es cero.  
2. Salida con media cercana a 0: reduce el sesgo de activación y acelera la convergencia.  
3. Aprendizaje más estable: especialmente útil en redes muy profundas.  
4. Mejor flujo del gradiente: las pendientes suaves evitan saturaciones bruscas.

---

### Desventajas

- Requiere el cálculo de $e^x$, lo que puede ser más costoso computacionalmente que ReLU.  
- Puede necesitar **normalización** o inicialización específica para rendir al máximo (por ejemplo, *He Normal*).

---

## - Inicializador de Pesos: He_Normal

El **inicializador He Normal** (o **Kaiming Initialization**) establece los pesos iniciales de cada capa densa usando una **distribución normal** con una desviación estándar dependiente del número de entradas.

### Definición matemática

$$
W \sim \mathcal{N}(0, \sqrt{\frac{2}{n_{\text{in}}}})
$$

donde:
- $W$ son los pesos de la capa.  
- $n_{\text{in}}$ es el número de entradas (neuronas) en la capa anterior.  
- $\mathcal{N}(\mu, \sigma)$ denota una **distribución normal** con media $\mu$ y desviación estándar $\sigma$.

---

### - ¿Por qué “He”?

Este método fue propuesto por **Kaiming He et al. (2015)**, y se diseñó especialmente para activaciones del tipo **ReLU** y **ELU**, que anulan parcialmente los valores negativos.

---

### - Motivación

En redes neuronales profundas, si los pesos se inicializan:
- Demasiado pequeños, los gradientes tienden a **desaparecer** (*vanishing gradient*).  
- Demasiado grandes, los gradientes pueden **explotar** (*exploding gradient*).

El inicializador **He Normal** mantiene la **varianza de las activaciones constante** a lo largo de las capas, lo que estabiliza el entrenamiento.

---

### Comparación con otros inicializadores

| Inicializador | Distribución | Recomendado para | Fórmula |
|:---------------|:-------------|:-----------------|:---------|
| **Glorot/Xavier** | Uniforme o normal | `tanh`, `sigmoid` | $$\sqrt{\frac{1}{n_{\text{in}} + n_{\text{out}}}}$$ |
| **He Normal** | Normal | `ReLU`, `ELU` | $$\sqrt{\frac{2}{n_{\text{in}}}}$$ |
| **LeCun Normal** | Normal | `SELU` | $$\sqrt{\frac{1}{n_{\text{in}}}}$$ |

---

## - Relación entre ELU y He Normal

Usar **ELU** junto con **He Normal** no es casualidad.  
Ambos se complementan para lograr entrenamientos más estables y rápida convergencia.

| Componente | Propósito |
|:------------|:-----------|
| `activation="elu"` | Asegura activaciones suaves y no saturadas |
| `kernel_initializer="he_normal"` | Mantiene gradientes estables a lo largo de las capas |
| **Combinación ELU + He Normal** | Evita gradientes inestables y mejora la eficiencia del entrenamiento |

---


In [ ]:
# Resumen del modelo - ¿Qué cambió?
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 3072)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100)            │       307,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_20 (Dense)                │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 500,210 (1.91 MB)

 Trainable params: 500,210 (1.91 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Compilación del modelo (no cambia nada)
model.compile(loss = "sparse_categorical_crossentropy", optimizer = "sgd", metrics = ["accuracy"])

In [ ]:
# Ejecuta este desastre
history.model.fit(x_train, y_train, epochs = 25, validation_data = (x_valid, y_valid), verbose = 0)

Epoch 1/25
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.5087 - loss: 1.3754 - val_accuracy: 0.4572 - val_loss: 1.5443
Epoch 2/25
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.5180 - loss: 1.3585 - val_accuracy: 0.4456 - val_loss: 1.5842
Epoch 3/25
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.5208 - loss: 1.3463 - val_accuracy: 0.4654 - val_loss: 1.5358
Epoch 4/25
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.5243 - loss: 1.3389 - val_accuracy: 0.4296 - val_loss: 1.6503
Epoch 5/25
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.5234 - loss: 1.3334 - val_accuracy: 0.4636 - val_loss: 1.5701
Epoch 6/25
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.5293 - loss: 1.3250 - val_accuracy: 0.4810 - val_loss: 1.5040
Epoch 7/25
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.5349 - loss: 1.3132 - val_accuracy: 0.4598 - val_loss: 1.5430
Epoch 8/25
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.5342 - loss: 1.3058 - 

In [ ]:
# Evaluando el nuevo modelo elu y he_normal
model.evaluate(x_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.1020 - loss: 181.5000


[180.44915771484375, 0.10379999876022339]

### 1.4 Fracaso rotundo, procedemos a compilar con optimizador Nadam y early stopping

In [ ]:
# Despeja el backend de Keras
keras.backend.clear_session()

# Semillas a 42
tf.random.set_seed(42)
np.random.seed(42)

# Armamos el modelo Secuencial
model = keras.models.Sequential()
model.add(keras.layers.Flatten(input_shape = [32,32,3]))
for _ in range(20):
    model.add(keras.layers.Dense(100, activation = "elu", kernel_initializer = "he_normal"))
model.add(keras.layers.Dense(10, activation = "softmax"))

## Optimizadores Adam y Nadam en Keras/TensorFlow

En el entrenamiento de redes neuronales, los **optimizadores** son algoritmos que actualizan los pesos del modelo para minimizar una función de pérdida.  
A continuación se analizan dos de los más usados: **Adam** y **Nadam**, ambos disponibles en `tensorflow.keras.optimizers`.

---

## Optimizador Adam (Adaptive Moment Estimation)

### Fundamento teórico

El optimizador Adam es un método de optimización de gradiente descendente estocástico que combina las ideas de **momentum** y **RMSProp** para adaptar la tasa de aprendizaje de cada parámetro individualmente.

Dado un conjunto de parámetros $\theta_t$ en el tiempo $t$, y un gradiente de pérdida:

$$
g_t = \nabla_\theta J(\theta_t)
$$

Adam calcula **dos momentos** del gradiente:

1. **Promedio móvil del primer momento (media):**

$$
m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t
$$

2. **Promedio móvil del segundo momento (varianza no centrada):**

$$
v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2
$$

Debido a que $m_t$ y $v_t$ se inicializan en cero, se corrigen para eliminar el sesgo inicial:

$$
\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad
\hat{v}_t = \frac{v_t}{1 - \beta_2^t}
$$

Finalmente, los parámetros se actualizan como:

$$
\theta_{t+1} = \theta_t - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}
$$

donde:
- $\eta$: tasa de aprendizaje (learning rate)
- $\epsilon$: pequeño valor para evitar división por cero
- $\beta_1$: controla el decaimiento del promedio del gradiente (momentum)
- $\beta_2$: controla el decaimiento del promedio de los gradientes al cuadrado (RMSProp)

---

### Parámetros de `keras.optimizers.Adam`

| Parámetro | Descripción | Valor típico |
|------------|--------------|--------------|
| `learning_rate` | Tasa de aprendizaje global $\eta$ | `0.001` |
| `beta_1` | Factor de decaimiento para el primer momento $m_t$ | `0.9` |
| `beta_2` | Factor de decaimiento para el segundo momento $v_t$ | `0.999` |
| `epsilon` | Pequeño valor para estabilidad numérica | `1e-7` |
| `amsgrad` | Usa la variante AMSGrad (opcional) | `False` |


In [ ]:
# Optimizador NADAM
optimizer = keras.optimizers.Nadam(learning_rate = 5e-5, beta_1 = 0.9, beta_2 = 0.999)
# Compila con NADAM
model.compile(loss = "sparse_categorical_crossentropy", optimizer = optimizer, metrics = ["accuracy"])

In [ ]:
# Early Stopping con callback a 20 puntos, guardadndo en modelocifar10.h5
early_stopping_cb = keras.callbacks.EarlyStopping(patience = 20)
model_checkpoint_cb = keras.callbacks.ModelCheckpoint("modelocifar10.h5", save_best_only = True)
callbacks = [early_stopping_cb, model_checkpoint_cb]

In [ ]:
# Entrena a 50 epochs
model.fit(x_train, y_train, epochs = 30, validation_data = (x_valid, y_valid), callbacks = callbacks, verbose = 0)

Epoch 1/30
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.1537 - loss: 9.6424

1407/1407 ━━━━━━━━━━━━━━━━━━━━ 18s 7ms/step - accuracy: 0.1537 - loss: 9.6383 - val_accuracy: 0.2430 - val_loss: 2.1226
Epoch 2/30
1402/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2495 - loss: 2.0670

1407/1407 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.2495 - loss: 2.0669 - val_accuracy: 0.2832 - val_loss: 1.9523
Epoch 3/30
1394/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2899 - loss: 1.9388

1407/1407 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.2900 - loss: 1.9386 - val_accuracy: 0.3192 - val_loss: 1.8640
Epoch 4/30
1402/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3190 - loss: 1.8522

1407/1407 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.3190 - loss: 1.8522 - val_accuracy: 0.3502 - val_loss: 1.7849
Epoch 5/30
1400/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3478 - loss: 1.7810

...

1407/1407 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.4724 - loss: 1.4731 - val_accuracy: 0.4304 - val_loss: 1.6026
Epoch 16/30
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.4788 - loss: 1.4517 - val_accuracy: 0.4308 - val_loss: 1.6074
Epoch 17/30
1398/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4844 - loss: 1.4367

1407/1407 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.4844 - loss: 1.4367 - val_accuracy: 0.4406 - val_loss: 1.5931
Epoch 18/30
1403/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4898 - loss: 1.4198

1407/1407 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.4898 - loss: 1.4198 - val_accuracy: 0.4386 - val_loss: 1.5873
Epoch 19/30
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.4959 - loss: 1.4056 - val_accuracy: 0.4366 - val_loss: 1.5942
Epoch 20/30
1396/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5018 - loss: 1.3910

1407/1407 ━━━━━━━━━━━━━━━━━━━━ 7s 5ms/step - accuracy: 0.5018 - loss: 1.3910 - val_accuracy: 0.4480 - val_loss: 1.5796
Epoch 21/30
1394/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5071 - loss: 1.3779

1407/1407 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.5071 - loss: 1.3779 - val_accuracy: 0.4498 - val_loss: 1.5784
Epoch 22/30
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.5106 - loss: 1.3640 - val_accuracy: 0.4364 - val_loss: 1.5990
Epoch 23/30
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.5172 - loss: 1.3531 - val_accuracy: 0.4382 - val_loss: 1.5980
Epoch 24/30
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.5211 - loss: 1.3439 - val_accuracy: 0.4396 - val_loss: 1.6094
Epoch 25/30
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.5248 - loss: 1.3293 - val_accuracy: 0.4422 - val_loss: 1.5984
Epoch 26/30
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.5283 - loss: 1.3218 - val_accuracy: 0.4528 - val_loss: 1.5874
Epoch 27/30
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.5296 - loss: 1.3083 - val_accuracy: 0.4542 - val_loss: 1.5915
Epoch 28/30
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - accuracy: 0.5368 - loss: 1.2950 - val_

In [ ]:
# Carga del callback y evaluación
model = keras.models.load_model("modelocifar10.h5")
model.evaluate(x_valid, y_valid)

157/157 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.4588 - loss: 1.5728


[1.5784395933151245, 0.4498000144958496]

### 1.5 Mucho mejor. Ahora agreguemos BN y comparemos los tiempos

In [ ]:
# Limpieza de sesión, semillas a 42
keras.backend.clear_session()
tf.random.set_seed(42)
np.random.seed(42)

# Batch Normalization (Normalización por Lotes)

## 1. Introducción

**Batch Normalization (BN)** es una técnica introducida por *Sergey Ioffe* y *Christian Szegedy* en 2015 para mejorar el entrenamiento de redes neuronales profundas.  
Su propósito principal es **estabilizar y acelerar el aprendizaje** al normalizar la activación de cada capa, reduciendo el problema del **"internal covariate shift"** (cambio en la distribución interna de los datos a lo largo del entrenamiento).

Durante el entrenamiento, las distribuciones de entrada a las capas intermedias cambian constantemente debido a la actualización de los pesos, lo cual dificulta que las capas siguientes aprendan de forma estable.  
Batch Normalization atenúa este problema normalizando las entradas de cada capa.

---

## 2. Operación matemática

Supongamos que se tiene un lote (batch) de activaciones intermedias de una capa:

$$
\{x_1, x_2, \dots, x_m\}
$$

donde  $m$ es el tamaño del batch.

### 2.1 Cálculo de la media y la varianza

Para cada característica (neurona o canal), se calcula la **media** y la **varianza** del lote:

$$
\mu_B = \frac{1}{m} \sum_{i=1}^{m} x_i
$$

$$
\sigma_B^2 = \frac{1}{m} \sum_{i=1}^{m} (x_i - \mu_B)^2
$$

### 2.2 Normalización

Cada activación del lote se normaliza restando la media y dividiendo por la desviación estándar:

$$
\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}
$$

donde $\epsilon$ es un pequeño valor positivo para evitar divisiones por cero.

### 2.3 Escalado y desplazamiento (reaprendizaje)

Luego, se introducen **parámetros aprendibles** $\gamma$ (escala) y $\beta$ (desplazamiento) que permiten que la red recupere la capacidad de representar distribuciones arbitrarias si lo necesita:

$$
y_i = \gamma \hat{x}_i + \beta
$$

Estos parámetros se aprenden junto con los pesos del modelo mediante retropropagación.

---


In [ ]:
# Armar modelo secuencial, capas Dense, Batchnorm y Activation
model = keras.models.Sequential()
model.add(keras.layers.Flatten(input_shape = [32,32,3]))
model.add(keras.layers.BatchNormalization())
for _ in range(20):
    model.add(keras.layers.Dense(100, kernel_initializer = "he_normal"))  # Capas sin activación z=Wx+b que entrega valores lineales
    model.add(keras.layers.BatchNormalization())
    model.add(keras.layers.Activation("elu"))
model.add(keras.layers.Dense(10, activation = "softmax"))

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 3072)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 3072)           │        12,288 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 100)            │       307,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 100)            │           400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 100)            │           400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 100)            │           400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 100)            │           400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 100)            │           400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 100)            │           400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼─────────────

 Total params: 520,498 (1.99 MB)

 Trainable params: 510,354 (1.95 MB)

 Non-trainable params: 10,144 (39.62 KB)

In [ ]:
# Optimizer nadam pero ahora si con beta1 y beta2
optimizer = keras.optimizers.Nadam(learning_rate = 5e-4, beta_1 =0.9, beta_2 = 0.999)
model.compile(loss = "sparse_categorical_crossentropy", optimizer = optimizer, metrics = ["accuracy"])

In [ ]:
# Early Sopping
early_stopping_cb = keras.callbacks.EarlyStopping(patience = 20)
model_checkpoint_cb = keras.callbacks.ModelCheckpoint("modelocifar10.h5", save_best_only = True)
callbacks = [early_stopping_cb, model_checkpoint_cb]

In [ ]:
#Fit el Modelo
model.fit(x_train, y_train, epochs = 30, validation_data = (x_valid, y_valid), callbacks = callbacks, verbose = 0)

Epoch 1/30
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.2909 - loss: 1.9871

1407/1407 ━━━━━━━━━━━━━━━━━━━━ 35s 11ms/step - accuracy: 0.2909 - loss: 1.9870 - val_accuracy: 0.4000 - val_loss: 1.6836
Epoch 2/30
1402/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.3996 - loss: 1.6876

1407/1407 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.3996 - loss: 1.6874 - val_accuracy: 0.4312 - val_loss: 1.6160
Epoch 3/30
1402/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4399 - loss: 1.5876

1407/1407 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.4399 - loss: 1.5876 - val_accuracy: 0.4450 - val_loss: 1.5639
Epoch 4/30
1404/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4656 - loss: 1.5132

1407/1407 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.4656 - loss: 1.5132 - val_accuracy: 0.4576 - val_loss: 1.5389
Epoch 5/30
1404/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4912 - loss: 1.4467

1407/1407 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.4912 - loss: 1.4467 - val_accuracy: 0.4610 - val_loss: 1.5383
Epoch 6/30
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.5151 - loss: 1.3863 - val_accuracy: 0.4572 - val_loss: 1.5599
Epoch 7/30
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.5363 - loss: 1.3334 - val_accuracy: 0.4562 - val_loss: 1.5897
Epoch 8/30
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.5536 - loss: 1.2865 - val_accuracy: 0.4584 - val_loss: 1.6065
Epoch 9/30
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 9s 6ms/step - accuracy: 0.5691 - loss: 1.2396 - val_accuracy: 0.4580 - val_loss: 1.6209
Epoch 10/30
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.5849 - loss: 1.1977 - val_accuracy: 0.4722 - val_loss: 1.6439
Epoch 11/30
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 9s 7ms/step - accuracy: 0.5982 - loss: 1.1602 - val_accuracy: 0.4700 - val_loss: 1.6616
Epoch 12/30
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.6128 - loss: 1.1238 - val_ac

In [ ]:
#Evaluar el Modelo
model = keras.models.load_model("modelocifar10.h5")
model.evaluate(x_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.4685 - loss: 1.5281


[1.532960295677185, 0.462799996137619]